In [ ]:
input_file = "/tmp/blobs.csv"                         
config_file = "config.json"

In [ ]:
import sys,os,ctypes
from pathlib import Path

#notebook_dir = os.getcwd()
#lib_path = os.path.join(notebook_dir,"../gng_py_lib/target/release")
#sys.path.insert(0,os.path.abspath(lib_path))
#print(lib_path)

import neurogas


import numpy as np
import pandas as pd
import time

In [ ]:
in_set = pd.read_csv(input_file)
in_set = in_set.to_numpy().flatten()

In [ ]:
ctx = gng_py.PyContext()
ctx = gng_py.PyContext()
ctx.load_config(config_file)
ctx.init_dataset_vec(in_set)
start = time.time()
ctx.fit()
end = time.time()
model_string = ctx.get_model_string()
print(end-start)

In [ ]:
import sys,os
import matplotlib.pyplot as plt
import json

In [ ]:
data = json.loads(model_string)

In [ ]:
points = None
edges = None
edge_positions = None
rows = []
for neuron in data["model"]["neurons"]:
    position = neuron["position"]
    id = neuron['id']
    x = position[0]
    y = position[1]
    rows.append({"id": id, "x": x, "y": y})

points = pd.DataFrame(rows)
points['class'] = points.index 

# Extract edges
row_edge = []
for edge in data['model']['edges']:
    edge_from = edge['from']
    edge_to = edge['to']
    row_edge.append({"start": edge_from, "to": edge_to})

edges = pd.DataFrame(row_edge)

# Calculate edge positions
row = []
for _, row_edge in edges.iterrows():
    try:
        row_from = points.loc[points['id'] == row_edge['start']].iloc[0]
        x_from = float(row_from['x'])
        y_from = float(row_from['y'])
        row_to = points.loc[points['id'] == row_edge['to']].iloc[0]
        x_to = float(row_to['x'])
        y_to = float(row_to['y'])

        row.append({"x_from": x_from, "y_from": y_from, "x_to": x_to, "y_to": y_to})
    except (IndexError, KeyError):
        continue

#edge_positions = pd.DataFrame(row)



In [ ]:
import matplotlib.pyplot as plt
plot = plt.figure()
ax = plot.add_subplot(111)

ax.clear()



# Plot points
ax.scatter(points['x'], points['y'],c=points['class'],cmap='hsv')

# Plot connections
for _, row in edges.iterrows():
    row_from = points.loc[points['id'] == row['start']].iloc[0]
    x_from = float(row_from['x'])
    y_from = float(row_from['y'])
    row_to = points.loc[points['id'] == row['to']].iloc[0]
    x_to = float(row_to['x'])
    y_to = float(row_to['y'])

    ax.plot([x_from, x_to], [y_from, y_to], 'k-', alpha=0.3)

In [ ]:
#from_class = []
#to_class = []
#for a in range(len(edges)):
#    given_index = a
#    curr_edge = edges.iloc[given_index].to_list()
#    point_class = []
#    given_id = curr_edge[0]
#    point_class.append(points[points["id"] == curr_edge[0]]["class"].iloc[0])
#    point_class.append(points[points["id"] == curr_edge[1]]["class"].iloc[0])
#    point_class
#    from_class.append(int(point_class[0]))
#    to_class.append(int(point_class[1]))


# Class calculation

In [ ]:
import numpy as np

def calc_classes_numpy(f_points, f_edges):
    # Extract columns as NumPy arrays
    point_ids = f_points["id"].to_numpy()
    point_class = f_points["class"].to_numpy()

    edge_start = f_edges["start"].to_numpy()
    edge_to    = f_edges["to"].to_numpy()

    num_points = len(point_ids)
    num_edges  = len(edge_start)

    # --- Mapping: point_id -> index ---
    # (Assuming ids are arbitrary integers)
    # Build fast lookup dictionary: id → index
    id_to_index = {pid: i for i, pid in enumerate(point_ids)}

    changed = True
    while changed:
        changed = False

        for p_idx in range(num_points):
            curr_point_id = point_ids[p_idx]

            # Loop edges
            for e in range(num_edges):

                s = edge_start[e]
                t = edge_to[e]

                # Case 1: point is edge start
                if s == curr_point_id:
                    i_s = id_to_index[s]
                    i_t = id_to_index[t]

                    if point_class[i_s] > point_class[i_t]:
                        point_class[i_s] = point_class[i_t]
                        changed = True
                        break

                # Case 2: point is edge end
                if t == curr_point_id:
                    i_s = id_to_index[t]
                    i_t = id_to_index[s]

                    if point_class[i_s] > point_class[i_t]:
                        point_class[i_s] = point_class[i_t]
                        changed = True
                        break

    # Write classes back to DataFrame
    f_points["class"] = point_class
    return f_points

In [ ]:
def calc_classes(f_points,f_edges):
    num_points = len(f_points)

    changed = True
    while changed:
        changed = False  # reset ONCE per iteration

        for p in range(num_points):
            curr_point_id = int(f_points.iloc[p]["id"])

            for _, edge in f_edges.iterrows():

                # Case 1: current point is the edge start
                if edge.start == curr_point_id:
                    class_start = f_points.loc[f_points.id == edge.start, "class"].iloc[0]
                    class_to =    f_points.loc[f_points.id == edge.to,    "class"].iloc[0]

                    if class_start > class_to:
                        f_points.loc[f_points.id == edge.start, "class"] = class_to
                        changed = True
                        break

                # Case 2: current point is the edge end
                if edge.to == curr_point_id:
                    class_start = f_points.loc[f_points.id == edge.to, "class"].iloc[0]
                    class_to =    f_points.loc[f_points.id == edge.start, "class"].iloc[0]

                    if class_start > class_to:
                        f_points.loc[f_points.id == edge.to, "class"] = class_to
                        changed = True
                        break
    return f_points    

In [ ]:
points = calc_classes_numpy(points,edges)

In [ ]:
points["class"].unique()

In [ ]:
import matplotlib.pyplot as plt
plot = plt.figure()
ax = plot.add_subplot(111)

ax.clear()



# Plot points
import matplotlib as mpl
from matplotlib.colors import ListedColormap

my_norm = mpl.colors.Normalize(vmin=points['class'].min(), 
                            vmax=points['class'].max())
my_map = ListedColormap(["red","blue"])
#ax.scatter(points['x'], points['y'],c=points['class'],cmap='hsv',norm=my_norm)
ax.scatter(points['x'], points['y'],c=points['class'],cmap=my_map,norm=my_norm)

# Plot connections
for _, row in edges.iterrows():
    row_from = points.loc[points['id'] == row['start']].iloc[0]
    x_from = float(row_from['x'])
    y_from = float(row_from['y'])
    row_to = points.loc[points['id'] == row['to']].iloc[0]
    x_to = float(row_to['x'])
    y_to = float(row_to['y'])

    ax.plot([x_from, x_to], [y_from, y_to], 'k-', alpha=0.3)
plt.show()